# Live-Streaming Indic Parler-TTS (Tamil / Tanglish) — Kaggle T4

End-to-end notebook: install → load model → text normalization → sentence chunking → **streaming** synthesis → live test.

**Voice:** Jaya (Tamil, female) — tuned for moderate enthusiasm, natural (not rushed) pace, high clarity.

**Design notes (read before running):**
- Streaming works because Indic Parler-TTS is *autoregressive* — audio is decoded token-by-token, so we can yield playback-ready chunks mid-generation instead of waiting for the full clip. We use a `ParlerTTSStreamer` for this.
- Chunking happens **once**, at the sentence level (paragraph → sentences). We do **not** split a sentence again into "normal words" vs "spelled-out words" as two separate TTS calls — instead, digits/codes/times are expanded into spoken word-form as plain text *before* the single TTS call per sentence. This avoids stitching seams between separately-generated clips.
- Numbers/codes are normalized via explicit `{{TAG:value}}` markers (recommended — your dialogue manager knows the semantic type of each number) with a regex-heuristic fallback for any untagged freeform text.
- Kaggle notebooks have no live telephony/speaker output — this notebook proves and measures **real incremental generation** (true streaming, not simulated) via timestamps and progressive `IPython.display.Audio` widgets per chunk. In production, each yielded chunk gets pushed over a WebSocket/RTP connection (e.g. via LiveKit Agents / Pipecat) instead of being collected here.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face login successful!")

## 1. Environment check — confirm T4 GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 2. Install dependencies

(`parler-tts` isn't on PyPI in a streaming-ready form — install from GitHub. `indic-num2words` gives us Tamil/English-Indian number-to-words with lakh/crore grouping. `transformers` is pinned deliberately — see the comment in the cell below.)

**After this cell finishes, restart the kernel** (Kaggle menu: Run → Restart Session, or the restart icon) before running anything below. Kaggle's base image already has a `transformers` import cached; pip-installing a different pinned version doesn't retroactively change what's already loaded in a live kernel process. Restart, then run all cells from the top again (the install cell is safe to re-run — pip will just confirm it's already satisfied).


In [ ]:
!pip install -q -U git+https://github.com/huggingface/parler-tts.git
# Pin transformers instead of using -U: parler-tts's logits_processors.py imports
# `isin_mps_friendly` from transformers.pytorch_utils, which only exists in a specific
# window of transformers releases. 4.46.1 is a known-good version for parler-tts.
!pip install -q "transformers==4.46.1" accelerate sentencepiece soundfile indic-numtowords

# Kaggle's base image ships protobuf 4.25.9, missing `runtime_version` (added in 5.27+),
# which transformers/sentencepiece need. Pin to a specific 5.x release rather than -U —
# jumping to whatever's newest (7.x) is untested against this stack and pulls in more
# resolver conflicts with Kaggle's preinstalled TensorFlow/Google-Cloud packages than needed.
!pip install -q "protobuf==5.29.5"

## 3. Imports & device setup

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import re
import time
import queue
import numpy as np
import soundfile as sf
import torch
from threading import Thread
from IPython.display import Audio, display
from transformers import AutoTokenizer
from transformers.generation.streamers import BaseStreamer
from parler_tts import ParlerTTSForConditionalGeneration
from indic_numtowords import num2words as indic_num2words

torch.manual_seed(42)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device.startswith("cuda") else torch.float32
print("Using device:", device, "| dtype:", torch_dtype)

## 4. Load Indic Parler-TTS

In [ ]:
MODEL_ID = "ai4bharat/indic-parler-tts"

model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    attn_implementation="eager",   # safest fast-attention choice on a T4; flash-attn-2 build can be flaky on Kaggle
).to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

SAMPLE_RATE = model.config.sampling_rate
print("Model loaded. Sample rate:", SAMPLE_RATE)

## 5. Streaming class

`ParlerTTSStreamer` decodes the audio codec in small windows as new tokens arrive, instead of waiting for the whole sequence. We try the library's built-in version first; if your installed `parler-tts` version doesn't expose it at the top level, we fall back to a local implementation (same logic as HuggingFace's reference streamer).

In [ ]:
try:
    from parler_tts import ParlerTTSStreamer
    print("Using parler_tts's built-in ParlerTTSStreamer")
except ImportError:
    print("Built-in streamer not importable — using local definition")

    class ParlerTTSStreamer(BaseStreamer):
        def __init__(self, model, device=None, play_steps=10, stride=None, timeout=None):
            self.model = model
            self.audio_encoder = model.audio_encoder
            self.generation_config = model.generation_config
            self.device = device if device is not None else model.device
            self.play_steps = play_steps

            hop_length = np.prod(self.audio_encoder.config.upsampling_ratios)
            self.stride = stride if stride is not None else hop_length * (play_steps - 1)

            self.token_cache = None
            self.to_yield = 0

            self.audio_queue = queue.Queue()
            self.stop_signal = None
            self.timeout = timeout

        def apply_delay_pattern_mask(self, input_ids):
            _, delay_pattern_mask = self.model.decoder.build_delay_pattern_mask(
                input_ids[:, :1],
                pad_token_id=self.generation_config.decoder_start_token_id,
                max_length=input_ids.shape[-1],
            )
            input_ids = self.model.decoder.apply_delay_pattern_mask(input_ids, delay_pattern_mask)
            input_ids = input_ids[:, 1:]
            first_valid = (input_ids[0, :] != self.generation_config.pad_token_id).nonzero()[0, 0]
            input_ids = input_ids[..., first_valid:]
            output_values = self.audio_encoder.decode(
                input_ids.unsqueeze(0).unsqueeze(0),
                audio_scales=[None],
            )
            audio_values = output_values.audio_values[0, 0]
            return audio_values.cpu().float().numpy()

        def put(self, value):
            batch_size = value.shape[0]
            if batch_size > 1:
                raise ValueError("ParlerTTSStreamer only supports batch size 1")
            elif len(value.shape) > 1:
                value = value[0, :, None]

            if self.token_cache is None:
                self.token_cache = value
            else:
                self.token_cache = torch.concatenate([self.token_cache, value[:, None]], dim=-1)

            if self.token_cache.shape[-1] % self.play_steps == 0:
                audio_values = self.apply_delay_pattern_mask(self.token_cache)
                self.on_finalized_audio(audio_values[self.to_yield: -self.stride])
                self.to_yield += len(audio_values) - self.to_yield - self.stride

        def end(self):
            if self.token_cache is not None:
                audio_values = self.apply_delay_pattern_mask(self.token_cache)
            else:
                audio_values = np.zeros(self.to_yield)
            self.on_finalized_audio(audio_values[self.to_yield:], stream_end=True)

        def on_finalized_audio(self, audio, stream_end=False):
            self.audio_queue.put(audio, timeout=self.timeout)
            if stream_end:
                self.audio_queue.put(self.stop_signal, timeout=self.timeout)

        def __iter__(self):
            return self

        def __next__(self):
            value = self.audio_queue.get(timeout=self.timeout)
            if not isinstance(value, np.ndarray) and value == self.stop_signal:
                raise StopIteration()
            return value

## 6. Voice profile — Jaya, moderate enthusiasm, clear, not rushed

Keep this description string **fixed** across calls — inconsistent descriptions are one of the main causes of Parler-family hallucination/instability.

In [ ]:
JAYA_VOICE_DESCRIPTION = (
    "Jaya speaks Tamil and English in a calm, steady, natural, and friendly tone, "
    "at a clear, consistent, composed pace that is not rushed and not too fast. "
    "Her volume is even and steady throughout, with pleasant, relaxed intonation. "
    "Her pronunciation is very clear and easy to understand. "
    "The recording is very high quality, close-up, and completely free of background noise."
)
print(JAYA_VOICE_DESCRIPTION)


In [ ]:
# Cache the voice-description text-encoder output.
#
# JAYA_VOICE_DESCRIPTION never changes between sentences, but stream_generate()
# was re-running it through the (~770M param) flan-t5-large text encoder on
# *every single* generate() call - one full encoder forward pass per sentence,
# purely to recompute a result that was already identical last time.
#
# ParlerTTSForConditionalGeneration.generate() accepts a precomputed
# `encoder_outputs=` kwarg and, when present, skips the text-encoder forward
# pass entirely (see `_prepare_text_encoder_kwargs_for_generation` in
# modeling_parler_tts.py - it only runs when "encoder_outputs" is NOT already
# in the kwargs). So we compute it once per distinct description text and
# reuse it. A *fresh* BaseModelOutput wrapper is constructed around the cached
# tensor on every call (rather than reusing the same object) because generate()
# can reassign `.last_hidden_state` on the object it's given internally when
# `prompt_cross_attention` is enabled - reusing the same instance across calls
# could let that mutation compound. Wrapping the same cached tensor freshly
# each time is nearly free and avoids that risk entirely.
from transformers.modeling_outputs import BaseModelOutput

_description_encoder_cache = {}

def get_cached_description_encoder_outputs(description_text: str):
    if description_text not in _description_encoder_cache:
        desc_ids = description_tokenizer(description_text, return_tensors="pt").to(device)
        with torch.inference_mode():
            hidden = model.get_text_encoder()(
                input_ids=desc_ids.input_ids,
                attention_mask=desc_ids.attention_mask,
                return_dict=True,
            ).last_hidden_state
            if (
                model.text_encoder.config.hidden_size != model.decoder.config.hidden_size
                and model.decoder.config.cross_attention_hidden_size is None
            ):
                hidden = model.enc_to_dec_proj(hidden)
            hidden = hidden * desc_ids.attention_mask[..., None]
        _description_encoder_cache[description_text] = hidden
    return _description_encoder_cache[description_text]

In [ ]:
# Warm-up call: first CUDA generation is always slow due to kernel compilation/allocation.
# Run this once so later latency numbers reflect steady-state performance, not cold start.
_warm_desc = description_tokenizer(JAYA_VOICE_DESCRIPTION, return_tensors="pt").to(device)
_warm_prompt = tokenizer("வணக்கம்.", return_tensors="pt").to(device)
with torch.inference_mode():
    _ = model.generate(
        input_ids=_warm_desc.input_ids,
        attention_mask=_warm_desc.attention_mask,
        prompt_input_ids=_warm_prompt.input_ids,
        prompt_attention_mask=_warm_prompt.attention_mask,
        max_new_tokens=50,
    )
torch.cuda.synchronize() if device.startswith("cuda") else None
print("Warm-up complete.")

## 7. Text normalization

Handles: Tamil, English, Tanglish mixing, digit-by-digit numbers (OTP/phone), currency/quantity numbers spoken in full (thousands/hundreds via `indic-num2words`, lakh/crore-aware), time expressions ("seven thirty"), and alphanumeric codes spelled letter-by-letter + digit-by-digit (vehicle numbers, booking IDs).

**Recommended usage — tag your text upstream:**
```
"Your OTP is {{OTP:483927}}."
"Fare is {{AMOUNT:1250}} rupees."
"Cab arrives at {{TIME:19:30}}."
"Vehicle number {{CODE:TN45AB1234}}."
```
Anything left untagged falls through to a regex-heuristic fallback, so nothing crashes or gets skipped if a tag is missed — but the tagged path is exact and should be your primary path in production.

In [ ]:
TAMIL_RE = re.compile(r'[\u0B80-\u0BFF]')

EN_DIGIT_WORDS = {'0': 'zero', '1': 'one', '2': 'two', '3': 'three', '4': 'four',
                   '5': 'five', '6': 'six', '7': 'seven', '8': 'eight', '9': 'nine'}

EN_LETTER_NAMES = {
    'A': 'A', 'B': 'bee', 'C': 'see', 'D': 'dee', 'E': 'ee',
    'F': 'eff', 'G': 'jee', 'H': 'aitch', 'I': 'eye', 'J': 'jay',
    'K': 'kay', 'L': 'ell', 'M': 'em', 'N': 'en', 'O': 'Oo',
    'P': 'pee', 'Q': 'Q', 'aar': 'ar', 'S': 'ess', 'T': 'tee',
    'U': 'you', 'V': 'vee', 'W': 'double-u', 'X': 'ex', 'Y': 'why', 'Z': 'zed'
}

def contains_tamil(text: str) -> bool:
    return bool(TAMIL_RE.search(text))

def spell_digits(digit_str: str) -> str:
    """Digit-by-digit reading in English words, comma-separated for a clear pause between each digit."""
    return ", ".join(EN_DIGIT_WORDS[d] for d in digit_str if d.isdigit())

def spell_alnum_code(code: str) -> str:
    """Spell alphanumeric codes using phonetic letter names.
    Guarantees 'BY' is spoken as 'bee, why' (never 'buy'), and 'AB' as 'ay, bee'."""
    parts = []
    for ch in code:
        if ch.upper() in EN_LETTER_NAMES:
            parts.append(EN_LETTER_NAMES[ch.upper()])
        elif ch in EN_DIGIT_WORDS:
            parts.append(EN_DIGIT_WORDS[ch])
    return ", ".join(parts)

def spell_time(hh: str, mm: str) -> str:
    """'19:30' -> 'seven thirty'; '07:05' -> 'seven oh five'; '18:00' -> 'six o'clock'."""
    h = int(hh) % 24
    m = int(mm)
    h12 = h % 12
    h12 = 12 if h12 == 0 else h12
    hour_word = indic_num2words(h12, lang="en")
    if m == 0:
        return f"{hour_word} o'clock"
    elif m < 10:
        return f"{hour_word} oh {indic_num2words(m, lang='en')}"
    else:
        return f"{hour_word} {indic_num2words(m, lang='en')}"

def spell_amount(num_str: str, lang: str = "en") -> str:
    """Full cardinal number in words, e.g. 350 -> 'three hundred fifty'.
    Always speaks numbers in English as requested."""
    cleaned = re.sub(r'[^\d.]', '', str(num_str)).split('.')[0].strip()
    if not cleaned:
        return num_str
    n = int(cleaned)
    return indic_num2words(n, lang="en")


In [ ]:
# --- Tag-based normalization (primary, recommended path) ---

TAG_RE = re.compile(r'\{\{(OTP|PHONE|AMOUNT|TIME|CODE|NUM):([^}]+)\}\}', re.IGNORECASE)
PRONUNCIATION_OVERRIDES = {
    "mins": "minutes",   # try the unabbreviated word first - abbreviations seem to confuse it
    "Journey": "ஜர்னி",   # Tamil-script transliteration - try this if "journey"/"Journey" keeps coming out wrong
}
PRONUNCIATION_OVERRIDE_RE = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in PRONUNCIATION_OVERRIDES) + r')\b',
    re.IGNORECASE,
)

def apply_pronunciation_overrides(text: str) -> str:
    def _replace(m):
        matched = m.group(0)
        for key, val in PRONUNCIATION_OVERRIDES.items():
            if key.lower() == matched.lower():
                return val
        return matched
    return PRONUNCIATION_OVERRIDE_RE.sub(_replace, text)


def expand_tag(match: "re.Match", lang: str = "en") -> str:
    kind, value = match.group(1).upper(), match.group(2).strip()
    if kind in ("OTP", "PHONE"):
        return spell_digits(value)
    elif kind in ("AMOUNT", "NUM"):
        # Always expand numbers in English
        return spell_amount(value, lang="en")
    elif kind == "TIME":
        hh, mm = value.split(":")
        return spell_time(hh, mm)
    elif kind == "CODE":
        # Strip any internal spaces or hyphens before spelling out
        clean_code = re.sub(r'[\s-]+', '', value)
        return spell_alnum_code(clean_code)
    return value

def normalize_tagged_text(text: str) -> str:
    def _sub(m):
        expansion = expand_tag(m, lang="en")
        if m.group(1).upper() in ("OTP", "PHONE", "CODE"):
            preceding = text[:m.start()].rstrip()
            if preceding and preceding[-1] not in ",:.!?":
                return ", " + expansion
        return expansion
    return TAG_RE.sub(_sub, text)

# --- Regex-heuristic fallback (safety net for untagged/freeform text) ---

STATE_CODES = "AN|AP|AR|AS|BR|CG|CH|DD|DL|DN|GA|GJ|HP|HR|JH|JK|KA|KL|LA|LD|MH|ML|MN|MP|MZ|NL|OD|OR|PB|PY|RJ|SK|TN|TR|TS|UA|UK|UP|WB"

# Matches Indian vehicle registrations (e.g. TN 07 BY 4321, DL 03 C 1111, TN07BY4321, 22 BH 1234 AA)
INDIAN_CAR_RE = re.compile(
    rf'\b({STATE_CODES})[ -]?([0-9]{{1,2}})[ -]?([A-Za-z]{{1,3}})?[ -]?([0-9]{{1,4}})\b',
    re.IGNORECASE
)
BH_CAR_RE = re.compile(r'\b([0-9]{2})[ -]?(BH)[ -]?([0-9]{4})[ -]?([A-Za-z]{1,2})\b', re.IGNORECASE)

ALNUM_CODE_RE = re.compile(
    r'\b(?=[A-Za-z0-9]{5,12}\b)(?=[A-Za-z0-9]*[A-Za-z])(?=[A-Za-z0-9]*\d)[A-Za-z0-9]{5,12}\b'
)
TIME_RE = re.compile(r'\b([01]?\d|2[0-3]):([0-5]\d)\b')
LONG_DIGIT_RE = re.compile(r'\b\d{4,}\b')   # phone/OTP-length runs -> digit-by-digit
SHORT_NUM_RE = re.compile(r'\b\d{1,3}\b')    # short bare numbers -> full cardinal word

def _replace_car_match(m: "re.Match") -> str:
    clean_car = re.sub(r'[\s-]+', '', m.group(0)).upper()
    return spell_alnum_code(clean_car)

def heuristic_normalize(text: str) -> str:
    # 1. Car numbers normalized first before digit extractors touch them
    text = INDIAN_CAR_RE.sub(_replace_car_match, text)
    text = BH_CAR_RE.sub(_replace_car_match, text)

    # 2. Standalone alphanumeric codes (e.g. Booking IDs)
    def _comma_before(pattern_sub_fn):
        def _sub(m):
            expansion = pattern_sub_fn(m.group(0))
            preceding = text[:m.start()].rstrip()
            if preceding and preceding[-1] not in ",:.!?":
                return ", " + expansion
            return expansion
        return _sub

    text = ALNUM_CODE_RE.sub(_comma_before(spell_alnum_code), text)
    text = LONG_DIGIT_RE.sub(_comma_before(spell_digits), text)

    # 3. Time, long digits, and short numbers
    text = TIME_RE.sub(lambda m: spell_time(m.group(1), m.group(2)), text)
    # text = LONG_DIGIT_RE.sub(lambda m: spell_digits(m.group(0)), text)
    text = SHORT_NUM_RE.sub(lambda m: spell_amount(m.group(0), lang="en"), text)
    return text

def normalize_text(text: str) -> str:
    text = apply_pronunciation_overrides(text)
    text = normalize_tagged_text(text)
    text = heuristic_normalize(text)
    return text

## 8. Sentence-level chunking

Paragraph → sentences (your original idea — kept). Very long sentences are further split at commas so no single chunk gets too long for stable, fast generation.

In [ ]:
# Fixed-width lookbehinds prevent splitting on periods after single-letter initials (e.g. "T. Nagar") or common honorifics
SENTENCE_SPLIT_RE = re.compile(
    r'(?<!\b[A-Za-z]\.)(?<!\bDr\.)(?<!\bMr\.)(?<!\bMs\.)(?<!\bMrs\.)(?<!\bSt\.)(?<!\bNo\.)(?<!\be\.g\.)(?<!\bi\.e\.)(?<=[.!?\u0964])\s+|\n+'
)

# Chunks longer than MAX_CHUNK_CHARS are split at natural punctuation or clause boundaries.
# IMPORTANT: this runs on RAW text before normalize_text() so alphanumeric codes are not shredded.
MAX_CHUNK_CHARS = 100
PUNCT_SPLIT_RE = re.compile(r'(?<=[,;:\-—])\s+')
CONJ_SPLIT_RE = re.compile(r'\s+(?=(?:and|but|or|because|while|after|before|மற்றும்|ஆனால்|எனவே)\b)', re.IGNORECASE)

def split_long_sentence(sentence: str, max_chars: int = MAX_CHUNK_CHARS):
    sentence = sentence.strip()
    if len(sentence) <= max_chars:
        return [sentence]

    protected_spans = [m.span() for m in TAG_RE.finditer(sentence)]
    def _inside_tag(pos):
        return any(start <= pos < end for start, end in protected_spans)

    # 1. Try splitting at punctuation (, ; : - —)
    pieces, last_end = [], 0
    for m in PUNCT_SPLIT_RE.finditer(sentence):
        if _inside_tag(m.start()):
            continue
        p = sentence[last_end:m.start()].strip().rstrip(',;:-— ')
        if p:
            pieces.append(p)
        last_end = m.end()
    final_p = sentence[last_end:].strip().rstrip(',;:-— ')
    if final_p:
        pieces.append(final_p)

    # 2. If no punctuation split was found, try splitting at conjunctions
    if len(pieces) <= 1:
        pieces, last_end = [], 0
        for m in CONJ_SPLIT_RE.finditer(sentence):
            if _inside_tag(m.start()):
                continue
            p = sentence[last_end:m.start()].strip()
            if p:
                pieces.append(p)
            last_end = m.start()
        final_p = sentence[last_end:].strip()
        if final_p:
            pieces.append(final_p)

    # 3. If still no split point, split at word spaces before max_chars
    if len(pieces) <= 1:
        words = sentence.split(" ")
        pieces, cur = [], ""
        for w in words:
            candidate = (cur + " " + w).strip() if cur else w
            if cur and len(candidate) > max_chars:
                pieces.append(cur)
                cur = w
            else:
                cur = candidate
        if cur:
            pieces.append(cur)

    # Greedily group pieces up to max_chars
    grouped, current = [], ""
    for piece in pieces:
        candidate = (current + " " + piece).strip() if current else piece
        if current and len(candidate) > max_chars:
            grouped.append(current)
            current = piece
        else:
            current = candidate
    if current:
        grouped.append(current)
    return grouped

def split_into_sentences(text: str):
    """Splits RAW text sentence-by-sentence at periods, ending punctuation, and newlines
    (protecting abbreviations like 'T. Nagar', 'Dr.', 'St.'), then further caps any
    resulting sentence that is still too long at natural clause boundaries.
    Call this BEFORE normalize_text(), not after."""
    sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(text.strip()) if s.strip()]
    chunks = []
    for s in sentences:
        chunks.extend(split_long_sentence(s))
    return chunks


## 9. Streaming generation + full pipeline

`stream_generate` runs `model.generate` in a background thread and yields audio chunks as they're decoded — this is real incremental generation, timestamped, not simulated.

In [ ]:
# --- Cell 22 ---
from transformers.modeling_outputs import BaseModelOutput

def stream_generate(prompt_text: str, description_text: str, play_steps_in_s: float = 0.5, max_new_tokens=None):
    frame_rate = model.audio_encoder.config.frame_rate
    play_steps = max(1, int(frame_rate * play_steps_in_s))

    streamer = ParlerTTSStreamer(model, device=device, play_steps=play_steps)

    prompt_clean = prompt_text.strip()
    if not prompt_clean.endswith(('.', '!', '?', '।')):
        prompt_clean += '.'

    prompt_ids = tokenizer(prompt_clean, return_tensors="pt").to(device)

    # min_new_tokens only blocks early <eos> - it doesn't generate more real speech.
    # Once actual content finishes before the floor is reached, the model is forced to
    # keep emitting *something*, which is what surfaces as noise/hallucinated words at
    # the tail. CHARS_PER_SEC=9.0 was overshooting broadly, so raised it (shorter floor)
    # AND added a nearby ceiling, so generation has a narrow window to naturally stop in
    # rather than either cutting off early or wandering past the real end.
    CHARS_PER_SEC = 13.0
    estimated_seconds = len(prompt_clean) / CHARS_PER_SEC
    min_new_tokens = max(10, int(frame_rate * estimated_seconds * 0.85))
    estimated_max_tokens = int(frame_rate * estimated_seconds * 1.6)

    cached_desc = get_cached_description_encoder_outputs(description_text)
    encoder_outputs = BaseModelOutput(last_hidden_state=cached_desc)

    gen_kwargs = dict(
        encoder_outputs=encoder_outputs,
        prompt_input_ids=prompt_ids.input_ids,
        prompt_attention_mask=prompt_ids.attention_mask,
        streamer=streamer,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.25,
        min_new_tokens=min_new_tokens,
        max_new_tokens=max_new_tokens if max_new_tokens else estimated_max_tokens,
    )

    def _run(**kwargs):
        with torch.inference_mode():
            model.generate(**kwargs)

    thread = Thread(target=_run, kwargs=gen_kwargs)
    start = time.time()
    thread.start()

    for chunk in streamer:
        yield chunk, time.time() - start

    thread.join()

In [ ]:
def speak(text: str, description: str = JAYA_VOICE_DESCRIPTION, play_steps_in_s: float = 0.5, verbose: bool = True):
    chunks_raw = split_into_sentences(text)
    chunks = [normalize_text(c) for c in chunks_raw]

    if verbose:
        print(f"Split into {len(chunks)} chunk(s):")
        for i, c in enumerate(chunks):
            print(f"  [{i+1}] {c}")

    full_audio = []
    overall_start = time.time()

    for i, chunk_text in enumerate(chunks):
        chunk_audio_pieces = []
        first_byte_time = None
        for audio_piece, t in stream_generate(chunk_text, description, play_steps_in_s=play_steps_in_s):
            if first_byte_time is None:
                first_byte_time = t
                if verbose:
                    print(f"  chunk {i+1}: first audio byte at {t:.3f}s")
            chunk_audio_pieces.append(audio_piece)

        chunk_audio = np.concatenate(chunk_audio_pieces) if chunk_audio_pieces else np.zeros(0)
        full_audio.append(chunk_audio)
        display(Audio(chunk_audio, rate=SAMPLE_RATE, autoplay=False))

    full_audio = np.concatenate(full_audio) if full_audio else np.zeros(0)
    total_time = time.time() - overall_start
    audio_duration = len(full_audio) / SAMPLE_RATE
    if verbose and audio_duration > 0:
        print(f"\nTotal generation time: {total_time:.2f}s for {audio_duration:.2f}s of audio "
              f"(real-time factor = {total_time/audio_duration:.2f}; <1.0 means faster than real-time)")

    return full_audio, SAMPLE_RATE

## 10. Test cases — covering your full spec

OTP/digits, currency numbers (hundreds/thousands), time, alphanumeric codes (vehicle number, booking ID), and a full Tanglish mixed sentence.

In [ ]:
test_otp = "Your OTP is {{OTP:483927}}. Please do not share this with anyone."

test_time_ta = "உங்கள் கார் வரும் நேரம் {{TIME:19:30}} மணிக்கு."

test_amount = "The total fare for this trip is {{AMOUNT:1250}} rupees."

test_codes = "Your vehicle number is {{CODE:TN45AB1234}} and your booking ID is {{CODE:AB23HH097}}."

test_full_tanglish = (
    "வணக்கம்! உங்கள் கேப் {{TIME:07:30}} மணிக்கு வரும். "
    "Booking ID {{CODE:AB23HH097}}. Fare {{AMOUNT:450}} rupees. "
    "Driver-oda phone number {{PHONE:9876543210}}."
)

for t in [test_otp, test_time_ta, test_amount, test_codes, test_full_tanglish]:
    print("=" * 90)
    speak(t)

## 11. Notes, limits, and what changes for production

- **What's real vs. simulated here:** the streaming generation itself is real — `ParlerTTSStreamer` genuinely yields audio in windows as the model decodes, and the timestamps printed above are true incremental latency, not simulated. What's Kaggle-only is *how* we consume the stream: we collect chunks into `IPython.display.Audio` widgets. In production, replace that consumption step with pushing each yielded chunk over a WebSocket/RTP frame to your transport layer (LiveKit Agents / Pipecat) as soon as it's yielded — the generation side does not need to change.
- **T4-specific:** fp16 + SDPA attention is the safe default; Flash-Attention 2 can be finicky to build on Kaggle's stock CUDA/T4 combo — only chase it if SDPA latency isn't good enough. `torch.inference_mode()` around generation avoids unnecessary autograd bookkeeping.
- **Voice stability:** if you still see occasional cutoffs/hallucination after switching to per-sentence chunking + fixed description, add a duration sanity check after each chunk (expected duration ≈ characters × a rough seconds-per-character constant for Tamil/English) and auto-retry that one chunk if it's a clear outlier — cheap insurance, not covered in this notebook to keep it focused.
- **Tag coverage:** extend the `{{TAG:value}}` vocabulary as your domain grows (e.g. `{{DISTANCE:4.2}}` for "4.2 km", `{{RATING:4.8}}` for driver ratings) rather than leaning on the heuristic fallback — tags stay exact, regex guessing doesn't.
- **Next step for you:** wire `stream_generate()`'s per-chunk output into a Pipecat/LiveKit Agents pipeline as the TTS stage, replacing the `Audio()` display calls with frame pushes to the transport.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pyngrok nest_asyncio

In [ ]:
# --- Cell 28 ---
import asyncio
import numpy as np
from threading import Thread
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

app = FastAPI()

async def async_stream_generate(prompt_text, description_text, play_steps_in_s=0.3):
    """Bridges synchronous stream_generate into an async generator for FastAPI."""
    loop = asyncio.get_event_loop()
    q: asyncio.Queue = asyncio.Queue()
    STOP = object()

    def producer():
        try:
            for audio_piece, t in stream_generate(prompt_text, description_text, play_steps_in_s):
                loop.call_soon_threadsafe(q.put_nowait, (audio_piece, t))
        finally:
            loop.call_soon_threadsafe(q.put_nowait, STOP)

    Thread(target=producer, daemon=True).start()

    while True:
        item = await q.get()
        if item is STOP:
            break
        yield item


@app.websocket("/ws")
async def ws_synthesize(websocket: WebSocket):
    await websocket.accept()
    await websocket.send_json({"type": "config", "sample_rate": SAMPLE_RATE})
    try:
        while True:
            msg = await websocket.receive_json()
            if msg.get("type") != "synthesize":
                continue

            text = msg.get("text", "")
            chunks_raw = split_into_sentences(text)
            chunks = [normalize_text(c) for c in chunks_raw]
            await websocket.send_json({"type": "chunks", "chunks": chunks})

            for i, chunk_text in enumerate(chunks):
                await websocket.send_json({"type": "chunk_start", "index": i, "text": chunk_text})
                async for audio_piece, t in async_stream_generate(chunk_text, JAYA_VOICE_DESCRIPTION, play_steps_in_s=0.3):
                    pcm16 = np.clip(audio_piece, -1.0, 1.0)
                    pcm16 = (pcm16 * 32767).astype(np.int16)
                    await websocket.send_bytes(pcm16.tobytes())
                await websocket.send_json({"type": "chunk_end", "index": i})

            await websocket.send_json({"type": "done"})
    except WebSocketDisconnect:
        print("Client disconnected")


In [ ]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()


nest_asyncio.apply()  # lets uvicorn's event loop run inside Kaggle's already-running Jupyter loop

conf.get_default().auth_token = user_secrets.get_secret("NGROK_AUTH_TOKEN")

PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()

for t in ngrok.get_tunnels():   # clear any stale tunnel if you re-run this cell
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(PORT, "http")
ws_url = tunnel.public_url.replace("https://", "wss://").replace("http://", "ws://") + "/ws"
print("Backend is live. Paste this into the frontend:")
print(ws_url)

# Frontend Code

In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8" />
    <title>Live Tamil / Tanglish Voice Agent — Demo</title>
    <style>
        :root {
            --bg: #0f1115;
            --panel: #171a21;
            --border: #262b36;
            --text: #e6e9ef;
            --muted: #8b93a7;
            --accent: #4f8cff;
            --good: #35c98f;
            --warn: #f5c453;
            --bad: #ef5b6f;
            --font: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
        }

        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            background: var(--bg);
            color: var(--text);
            font-family: var(--font);
            min-height: 100vh;
        }

        header {
            padding: 20px 28px;
            border-bottom: 1px solid var(--border);
            display: flex;
            align-items: center;
            justify-content: space-between;
            flex-wrap: wrap;
            gap: 12px;
        }

        header h1 {
            font-size: 18px;
            font-weight: 600;
            margin: 0;
        }

        header h1 span {
            color: var(--muted);
            font-weight: 400;
        }

        .conn-bar {
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .conn-bar input {
            background: var(--panel);
            border: 1px solid var(--border);
            color: var(--text);
            padding: 8px 10px;
            border-radius: 6px;
            font-size: 13px;
            width: 340px;
        }

        .btn {
            background: var(--accent);
            border: none;
            color: white;
            padding: 8px 16px;
            border-radius: 6px;
            font-size: 13px;
            font-weight: 600;
            cursor: pointer;
        }

        .btn:disabled {
            opacity: 0.4;
            cursor: not-allowed;
        }

        .btn.secondary {
            background: var(--panel);
            border: 1px solid var(--border);
        }

        .status-dot {
            width: 9px;
            height: 9px;
            border-radius: 50%;
            background: var(--bad);
            display: inline-block;
            margin-right: 6px;
            box-shadow: 0 0 8px var(--bad);
        }

        .status-dot.connected {
            background: var(--good);
            box-shadow: 0 0 8px var(--good);
        }

        .status-dot.streaming {
            background: var(--warn);
            box-shadow: 0 0 8px var(--warn);
            animation: pulse 1s infinite;
        }

        @keyframes pulse {

            0%,
            100% {
                opacity: 1;
            }

            50% {
                opacity: 0.35;
            }
        }

        main {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
            padding: 24px 28px;
            max-width: 1300px;
            margin: 0 auto;
        }

        @media (max-width: 900px) {
            main {
                grid-template-columns: 1fr;
            }
        }

        .panel {
            background: var(--panel);
            border: 1px solid var(--border);
            border-radius: 10px;
            padding: 18px;
        }

        .panel h2 {
            font-size: 13px;
            text-transform: uppercase;
            letter-spacing: 0.06em;
            color: var(--muted);
            margin: 0 0 12px 0;
        }

        textarea {
            width: 100%;
            min-height: 160px;
            background: #0d0f14;
            border: 1px solid var(--border);
            color: var(--text);
            border-radius: 8px;
            padding: 12px;
            font-size: 14px;
            line-height: 1.5;
            resize: vertical;
            font-family: var(--font);
        }

        .controls {
            display: flex;
            align-items: center;
            gap: 10px;
            margin-top: 12px;
        }

        .viz {
            height: 48px;
            display: flex;
            align-items: center;
            gap: 3px;
            margin-top: 16px;
            padding: 0 4px;
        }

        .viz .bar {
            flex: 1;
            background: var(--accent);
            border-radius: 2px;
            height: 4px;
            transition: height 0.06s ease-out;
        }

        .stats {
            margin-top: 14px;
            display: flex;
            gap: 22px;
            font-size: 12px;
            color: var(--muted);
        }

        .stats b {
            color: var(--text);
            font-size: 14px;
            display: block;
        }

        .chunk-list {
            display: flex;
            flex-direction: column;
            gap: 8px;
            max-height: 480px;
            overflow-y: auto;
        }

        .chunk {
            border: 1px solid var(--border);
            border-radius: 8px;
            padding: 10px 12px;
            font-size: 13px;
            line-height: 1.45;
            background: #0d0f14;
            display: flex;
            gap: 10px;
            align-items: flex-start;
        }

        .chunk .idx {
            flex: none;
            width: 22px;
            height: 22px;
            border-radius: 50%;
            background: var(--border);
            color: var(--muted);
            font-size: 11px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 700;
        }

        .chunk.pending {
            opacity: 0.55;
        }

        .chunk.speaking {
            border-color: var(--warn);
            background: #1c1a12;
        }

        .chunk.speaking .idx {
            background: var(--warn);
            color: #1c1a12;
        }

        .chunk.done {
            border-color: var(--good);
        }

        .chunk.done .idx {
            background: var(--good);
            color: #06251a;
        }

        .chunk .text {
            flex: 1;
        }

        .empty-hint {
            color: var(--muted);
            font-size: 13px;
        }

        .preview-box {
            margin-top: 10px;
            padding: 10px 12px;
            background: #090a0d;
            border: 1px solid var(--border);
            border-radius: 6px;
            font-size: 12px;
        }

        .preview-header {
            display: flex;
            align-items: center;
            justify-content: space-between;
            margin-bottom: 6px;
            color: var(--muted);
            font-weight: 500;
        }

        .preview-badges {
            display: flex;
            gap: 6px;
            flex-wrap: wrap;
        }

        .tag-pill {
            display: inline-flex;
            align-items: center;
            padding: 2px 7px;
            border-radius: 4px;
            font-size: 11px;
            font-weight: 600;
            letter-spacing: 0.02em;
        }

        .tag-pill.amt {
            background: rgba(53, 201, 143, 0.15);
            color: #35c98f;
            border: 1px solid rgba(53, 201, 143, 0.3);
        }

        .tag-pill.otp {
            background: rgba(245, 196, 83, 0.15);
            color: #f5c453;
            border: 1px solid rgba(245, 196, 83, 0.3);
        }

        .tag-pill.phone {
            background: rgba(79, 140, 255, 0.15);
            color: #4f8cff;
            border: 1px solid rgba(79, 140, 255, 0.3);
        }

        .tag-pill.time {
            background: rgba(186, 104, 200, 0.15);
            color: #ba68c8;
            border: 1px solid rgba(186, 104, 200, 0.3);
        }

        .tag-pill.code {
            background: rgba(255, 138, 101, 0.15);
            color: #ff8a65;
            border: 1px solid rgba(255, 138, 101, 0.3);
        }

        .preview-text {
            color: #b8c0d4;
            font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
            font-size: 12px;
            line-height: 1.45;
            word-break: break-word;
            white-space: pre-wrap;
            max-height: 90px;
            overflow-y: auto;
        }

        .continuity-badge {
            display: inline-flex;
            align-items: center;
            gap: 5px;
            font-size: 11px;
            color: var(--good);
            background: rgba(53, 201, 143, 0.1);
            padding: 3px 8px;
            border-radius: 4px;
            border: 1px solid rgba(53, 201, 143, 0.25);
        }
    </style>
</head>

<body>

    <header>
        <h1>Live Voice Agent Demo <span>— Tamil / Tanglish streaming TTS</span></h1>
        <div class="conn-bar">
            <span class="continuity-badge">⚡ Gapless Audio Engine Active</span>
            <span><span id="statusDot" class="status-dot"></span><span id="statusText">Disconnected</span></span>
            <input id="wsUrl" type="text" placeholder="wss://skyrocket-rickety-hanky.ngrok-free.dev/ws" />
            <button class="btn secondary" id="connectBtn">Connect</button>
        </div>
    </header>

    <main>
        <div class="panel">
            <h2>Input text</h2>
            <textarea id="inputText"
                placeholder="Paste natural mixed Tamil / English / Tanglish paragraph here (e.g. 'The total fare is 1250 rupees, OTP is 483927, cab arrives at 07:30. Driver mobile 9876543210. Booking ID AB23HH097'). Numbers and keywords will be auto-tagged!"></textarea>

            <div class="preview-box" id="previewBox" style="display: none;">
                <div class="preview-header">
                    <span>✨ Auto-Tagged Text (Sent to Backend):</span>
                    <div class="preview-badges" id="previewBadges"></div>
                </div>
                <div class="preview-text" id="previewText"></div>
            </div>

            <div class="controls">
                <button class="btn" id="synthBtn" disabled>Synthesize &amp; Speak</button>
                <button class="btn secondary" id="playEntireBtn" disabled>Play entire audio</button>
                <span id="synthHint" class="empty-hint"></span>
            </div>

            <div class="viz" id="viz"></div>

            <div class="stats">
                <div><b id="statFirstByte">—</b>time to first audio</div>
                <div><b id="statChunks">—</b>sentence chunks</div>
                <div><b id="statTotal">—</b>total time</div>
            </div>
        </div>

        <div class="panel">
            <h2>Sentence-wise chunking (reviewer view)</h2>
            <div class="chunk-list" id="chunkList">
                <div class="empty-hint">Chunks will appear here once you synthesize something — each one highlights
                    while it's being spoken.</div>
            </div>
        </div>
    </main>

    <script>
        const statusDot = document.getElementById('statusDot');
        const statusText = document.getElementById('statusText');
        const wsUrlInput = document.getElementById('wsUrl');
        const connectBtn = document.getElementById('connectBtn');
        const synthBtn = document.getElementById('synthBtn');
        const playEntireBtn = document.getElementById('playEntireBtn');
        const inputText = document.getElementById('inputText');
        const previewBox = document.getElementById('previewBox');
        const previewText = document.getElementById('previewText');
        const previewBadges = document.getElementById('previewBadges');
        const chunkList = document.getElementById('chunkList');
        const vizEl = document.getElementById('viz');
        const statFirstByte = document.getElementById('statFirstByte');
        const statChunks = document.getElementById('statChunks');
        const statTotal = document.getElementById('statTotal');

        let ws = null;
        let audioCtx = null;
        let analyser = null;
        let sampleRate = 44100;
        let currentChunkIndex = -1;
        let chunkLastNode = {};
        let synthStart = 0;
        let firstByteRecorded = false;

        // --- Seamless gap-minimized playback config ---
        const CROSSFADE_SEC = 0.025; // 25ms crossfade between consecutive chunks
        const CADENCE_GAP_SEC = 0.05; // 50ms tight natural pause between sentences (eliminates 1-2s awkward silence)
        let nextBoundary = 0;        // the nominal start time for the next sentence
        let lastGainNode = null;     // previous sentence's GainNode for smooth crossfades
        let lastBufferDuration = 0;  // previous sentence's trimmed duration
        let currentChunkPieces = []; // Float32Array pieces received for the active sentence
        let scheduledTimeouts = [];  // tracks pending highlight timeouts

        // --- Stored sentence audio buffers for "Play Entire Audio" ---
        let storedSentences = [];    // Float32Array per sentence chunk
        let entireAudioSource = null; // active BufferSourceNode for whole-paragraph playback
        let entireAudioTimeouts = []; // highlight timeouts during whole-paragraph playback

        // --- visualizer bars ---
        const NUM_BARS = 40;
        for (let i = 0; i < NUM_BARS; i++) {
            const b = document.createElement('div');
            b.className = 'bar';
            vizEl.appendChild(b);
        }
        const bars = vizEl.querySelectorAll('.bar');

        function animateViz() {
            requestAnimationFrame(animateViz);
            if (!analyser) return;
            const data = new Uint8Array(analyser.frequencyBinCount);
            analyser.getByteFrequencyData(data);
            const step = Math.floor(data.length / NUM_BARS);
            for (let i = 0; i < NUM_BARS; i++) {
                const v = data[i * step] || 0;
                bars[i].style.height = Math.max(4, (v / 255) * 44) + 'px';
            }
        }
        animateViz();

        function setStatus(state) {
            statusDot.className = 'status-dot' + (state === 'connected' ? ' connected' : state === 'streaming' ? ' streaming' : '');
            statusText.textContent = state === 'connected' ? 'Connected' : state === 'streaming' ? 'Streaming' : 'Disconnected';
            synthBtn.disabled = state === 'disconnected';
        }

        function ensureAudioContext() {
            if (!audioCtx) {
                audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                analyser = audioCtx.createAnalyser();
                analyser.fftSize = 128;
                analyser.connect(audioCtx.destination);
                nextBoundary = audioCtx.currentTime;
            } else if (audioCtx.state === 'suspended') {
                audioCtx.resume();
            }
        }

        function containsTamil(text) {
            return /[\u0B80-\u0BFF]/.test(text);
        }

        /**
         * Auto-converts incoming natural paragraph by identifying keywords/patterns
         * and wrapping them in the backend tags: {{AMOUNT:...}}, {{OTP:...}},
         * {{PHONE:...}}, {{TIME:HH:MM}}, {{CODE:...}}
         * Also fixes phonetic stumbling on acronyms like OTP, UPI, and Booking ID.
         */
        function autoTagText(text) {
            if (!text || !text.trim()) return '';

            const placeholders = [];
            function saveTag(val) {
                placeholders.push(val);
                return '___AUTOTAG_' + (placeholders.length - 1) + '___';
            }

            // 0. Preserve any existing {{TAG:val}} and normalize tag name to uppercase
            text = text.replace(/\{\{([A-Za-z]+):([^}]+)\}\}/g, (match, tag, val) => {
                return saveTag('{{' + tag.toUpperCase() + ':' + val.trim() + '}}');
            });

            // 1. Durations (minutes/seconds/hours): e.g. "5:00 minutes" or "5 minutes"
            // Fix: "5:00 minutes" must become "5 minutes", NOT time of day like "5:00 o'clock"!
            const durationRegex = /\b(\d+)(?::00)?\s*(minutes?|mins?|sec(?:onds?)?|hrs?|hours?|நிமிடங்கள்?|நிமிடம்|வினாடிகள்?|மணி)(?:-ல்)?\b/gi;
            text = text.replace(durationRegex, (match, num, unit) => {
                return saveTag('{{AMOUNT:' + num + '}}') + ' ' + unit + (match.endsWith('-ல்') ? '-ல்' : '');
            });

            // 2. TIME of day: e.g. 19:30, 07:30, 7:30 pm (must NOT match durations like minutes)
            text = text.replace(/\b(0?[0-9]|1[0-9]|2[0-3]):([0-5][0-9])(?:\s*(am|pm))?\b(?!\s*(?:min|sec|hour|நிமி))/gi, (match, hStr, mm, ampm) => {
                let hh = parseInt(hStr, 10);
                if (ampm) {
                    ampm = ampm.toLowerCase();
                    if (ampm === 'pm' && hh < 12) hh += 12;
                    if (ampm === 'am' && hh === 12) hh = 0;
                }
                const hhStr = hh < 10 ? '0' + hh : '' + hh;
                return saveTag('{{TIME:' + hhStr + ':' + mm + '}}');
            });

            // 2.5 Place name abbreviations & titles: e.g. "T. Nagar" -> "T Nagar", "Dr. " -> "Dr "
            // Prevents sentence chunking from splitting on the abbreviation period
            text = text.replace(/\b([A-Z])\.\s+([A-Z][a-z]+)\b/g, '$1 $2');
            text = text.replace(/\b(Dr|Mr|Mrs|Ms|Prof|St|No|Rd|Ave)\.\s+/gi, '$1 ');

            // 3. INDIAN VEHICLE REGISTRATION NUMBERS (Standard & Bharat BH series):
            // Matches state code (TN, KA, DL, MH, etc.) + RTO + optional series + registration digits
            // with or without spaces/hyphens (e.g. TN07BY4321, TN 07 BY 4321, TN-07-BY-4321, DL 03 C 1111)
            // or BH series (e.g. 22 BH 1234 AA).
            // Plain "cab" is NEVER matched as a car number!
            const STATE_CODES = "AN|AP|AR|AS|BR|CG|CH|DD|DL|DN|GA|GJ|HP|HR|JH|JK|KA|KL|LA|LD|MH|ML|MN|MP|MZ|NL|OD|OR|PB|PY|RJ|SK|TN|TR|TS|UA|UK|UP|WB";
            const indianCarRegex = new RegExp(
                `\\b(${STATE_CODES})[ -]?([0-9]{1,2})[ -]?([A-Za-z]{1,3})?[ -]?([0-9]{1,4})\\b`,
                "gi"
            );
            const bhCarRegex = /\b([0-9]{2})[ -]?(BH)[ -]?([0-9]{4})[ -]?([A-Za-z]{1,2})\b/gi;

            // Optional keyword prefix before car number: e.g. "Car number: TN07BY4321", "Cab number TN 07 BY 4321"
            const carPrefixRegex = /\b((?:car(?:\s*number)?|vehicle(?:\s*number)?|cab\s*number|வண்டி\s*எண்|பதிவு\s*எண்)\s*(?:is|:|[-])?\s*)/gi;

            text = text.replace(carPrefixRegex, (match, prefix) => {
                return prefix.trim().replace(/[:\-]+$/, '') + ': ';
            });

            text = text.replace(indianCarRegex, (match) => {
                const cleanCar = match.replace(/[\s-]+/g, '').toUpperCase();
                return saveTag('{{CODE:' + cleanCar + '}}');
            });
            text = text.replace(bhCarRegex, (match) => {
                const cleanCar = match.replace(/[\s-]+/g, '').toUpperCase();
                return saveTag('{{CODE:' + cleanCar + '}}');
            });

            // 3b. BOOKING ID / ORDER ID / PNR / REF CODES:
            // Requires explicit ID keywords (does not trigger on plain words like "cab" or "booking")
            const codePrefixRegex = /\b((?:booking\s*id|order\s*id|pnr|ref(?:erence)?\s*id)\s*(?:is|:|[-])?\s*)([A-Za-z0-9]{3,14})\b/gi;
            text = text.replace(codePrefixRegex, (match, prefix, code) => {
                prefix = prefix.replace(/\bID\b/gi, 'I-D').trim().replace(/[:\-]+$/, '') + ': ';
                return prefix + saveTag('{{CODE:' + code.toUpperCase() + '}}');
            });

            // Standalone alphanumeric code with mixed digits and letters (e.g. TN07BY4321, AB849201)
            text = text.replace(/\b(?=[A-Za-z0-9]{5,12}\b)(?=[A-Za-z0-9]*[A-Za-z])(?=[A-Za-z0-9]*\d)[A-Za-z0-9]{5,12}\b/g, (match) => {
                return saveTag('{{CODE:' + match.toUpperCase() + '}}');
            });

            // 4. OTP / PIN keywords: e.g. "OTP 5931", "OTP is 483927", "pin 1234"
            // Insert a colon after keyword (e.g. 'O-T-P: ') to create a clean natural gap before reading digits
            const otpRegex = /\b((?:otp|pin|passcode|verification\s*code|one\s*time\s*password|ரகசிய\s*எண்|கடவுச்சொல்)\s*(?:is|:|[-])?\s*)(\d{4,8})\b/gi;
            text = text.replace(otpRegex, (match, prefix, digits) => {
                prefix = prefix.replace(/\bOTP\b/gi, 'O-T-P').trim().replace(/[:\-]+$/, '') + ', ';
                return prefix + saveTag('{{OTP:' + digits + '}}');
            });
            // 5. Phone keywords & numbers: e.g. "Phone Number: 9876543210"
            const phoneRegex = /\b((?:phone(?:\s*number)?|mobile(?:\s*number)?|call|contact|cell|tel|whatsapp|ph|தொடர்புக்கு|தொலைபேசி|அழைக்க)\s*(?:is|:|[-])?\s*)(?:\+?91[\s-]?)?([6-9]\d{9})\b/gi;
            text = text.replace(phoneRegex, (match, prefix, num) => {
                prefix = prefix.trim().replace(/[:\-]+$/, '') + ': ';
                return prefix + saveTag('{{PHONE:' + num + '}}');
            });

            // Standalone 10-digit Indian mobile number
            text = text.replace(/\b(?:\+91[\s-]?)?([6-9]\d{9})\b/g, (match, num) => {
                return saveTag('{{PHONE:' + num + '}}');
            });

            // 6. Currency / Amount:
            // Prefix symbols: ₹1250, Rs. 1250, $50
            text = text.replace(/(?:₹|rs\.?|inr|\$|usd|ரூபாய்|ரூ\.?)\s*(\d[\d,]*(?:\.\d+)?)\b/gi, (match, num) => {
                const clean = num.replace(/,/g, '');
                return saveTag('{{AMOUNT:' + clean + '}}') + ' rupees';
            });

            // Suffix currency words: 350 Rupees, 450 rs, 500 ரூபாய்
            text = text.replace(/\b(\d[\d,]*(?:\.\d+)?)\s*(rupees|rs\.?|inr|usd|ரூபாய்)/gi, (match, num, unit) => {
                const clean = num.replace(/,/g, '');
                return saveTag('{{AMOUNT:' + clean + '}}') + ' ' + unit;
            });

            // Amount keywords: Cost 350, fare 450, total 1250
            const amtRegex = /\b((?:amount|fare|price|cost|bill|fee|rent|total|balance|paid|charge|charges|due|pay|payment|cash|discount|refund|tip|tax|subtotal|விலை|கட்டணம்|தொகை|வாடகை|செலவு|பணம்)\s*(?:is|of|:|[-])?\s*)(\d[\d,]*(?:\.\d+)?)\b/gi;
            text = text.replace(amtRegex, (match, prefix, num) => {
                const clean = num.replace(/,/g, '');
                prefix = prefix.trim().replace(/[:\-]+$/, '') + ': ';
                return prefix + saveTag('{{AMOUNT:' + clean + '}}');
            });

            // 7. Any other remaining bare numbers -> convert to {{AMOUNT:...}}
            text = text.replace(/\b\d[\d,]*(?:\.\d+)?\b/g, (match) => {
                const clean = match.replace(/,/g, '');
                return saveTag('{{AMOUNT:' + clean + '}}');
            });

            // 8. Pronunciation helper for standalone acronyms:
            // "OTP" -> "O-T-P" so TTS says "Oh Tee Pee" instead of "OTAP"
            text = text.replace(/\bOTP\b/g, 'O-T-P');
            // "UPI" -> "U-P-I" so TTS says "You Pee Eye"
            text = text.replace(/\bUPI\b/g, 'U-P-I');
            // "ID" -> "I-D" so TTS says "Eye Dee" instead of "kid"
            text = text.replace(/\bID\b/g, 'I-D');

            // Restore saved tags
            for (let i = 0; i < placeholders.length; i++) {
                text = text.replace('___AUTOTAG_' + i + '___', placeholders[i]);
            }

            return text.replace(/\s{2,}/g, ' ').trim();
        }

        /**
         * Update the live preview box below the textarea
         */
        function updateTagPreview() {
            const raw = inputText.value.trim();
            if (!raw) {
                previewBox.style.display = 'none';
                return;
            }

            const tagged = autoTagText(raw);
            previewText.textContent = tagged;

            // Extract tags for pills
            const tagsFound = [...tagged.matchAll(/\{\{([A-Z]+):([^}]+)\}\}/g)];
            if (tagsFound.length > 0) {
                previewBox.style.display = 'block';
                previewBadges.innerHTML = '';
                const seen = new Set();
                tagsFound.forEach(m => {
                    const kind = m[1];
                    const val = m[2];
                    const key = kind + ':' + val;
                    if (seen.has(key)) return;
                    seen.add(key);
                    const pill = document.createElement('span');
                    pill.className = 'tag-pill ' + kind.toLowerCase();
                    pill.textContent = `${kind}: ${val}`;
                    previewBadges.appendChild(pill);
                });
            } else {
                previewBox.style.display = 'none';
            }
        }

        inputText.addEventListener('input', updateTagPreview);

        /**
         * Audio silence trimmer:
         * Trims only leading onset dead air to minimize playback latency.
         * NEVER trims the tail of the sentence so trailing words, soft consonants,
         * and natural breath decays are 100% preserved.
         */
        function trimAudioSilence(float32, sampleRate = 44100, threshold = 0.0008, padMs = 80) {
            const padSamples = Math.floor((padMs / 1000) * sampleRate);
            const windowSize = 256;
            const len = float32.length;
            if (len <= windowSize) return float32;

            // Find head onset (start of actual speech) to eliminate initial dead-air latency
            let start = 0;
            for (let i = 0; i < len - windowSize; i += 64) {
                let sumSq = 0;
                for (let j = 0; j < windowSize; j++) {
                    sumSq += float32[i + j] * float32[i + j];
                }
                const rms = Math.sqrt(sumSq / windowSize);
                if (rms > threshold) {
                    start = Math.max(0, i - padSamples);
                    break;
                }
            }

            // DO NOT trim the tail of the sentence. Autoregressive TTS stops on <eos>.
            // Tail RMS scanning chops off soft final syllables (e.g. "...rupees", "...street", "...nagar").
            const end = len;

            if (start >= end) return float32;

            const trimmed = float32.slice(start, end);
            // 5ms micro fade-in at the head to guarantee click-free onset without attenuating speech
            const fadeSamples = Math.min(Math.floor(0.005 * sampleRate), Math.floor(trimmed.length / 8));
            for (let i = 0; i < fadeSamples; i++) {
                trimmed[i] *= (i / fadeSamples);
            }
            return trimmed;
        }

        function connect() {
            const url = wsUrlInput.value.trim();
            if (!url) { alert('Paste the wss:// URL printed by your Kaggle notebook first.'); return; }
            ensureAudioContext();

            ws = new WebSocket(url);
            ws.binaryType = 'arraybuffer';

            ws.onopen = () => setStatus('connected');
            ws.onclose = () => setStatus('disconnected');
            ws.onerror = () => setStatus('disconnected');

            ws.onmessage = (event) => {
                if (typeof event.data === 'string') {
                    const msg = JSON.parse(event.data);
                    handleControlMessage(msg);
                } else {
                    handleAudioFrame(event.data);
                }
            };
        }

        function handleControlMessage(msg) {
            if (msg.type === 'config') {
                sampleRate = msg.sample_rate;
            } else if (msg.type === 'chunks') {
                renderChunkList(msg.chunks);
                statChunks.textContent = msg.chunks.length;
                storedSentences = new Array(msg.chunks.length).fill(null);
                playEntireBtn.disabled = true;
                setStatus('streaming');
            } else if (msg.type === 'chunk_start') {
                currentChunkIndex = msg.index;
                currentChunkPieces = [];
            } else if (msg.type === 'chunk_end') {
                scheduleSentence(msg.index);
            } else if (msg.type === 'done') {
                setStatus('connected');
                statTotal.textContent = ((performance.now() - synthStart) / 1000).toFixed(2) + 's';
                if (storedSentences.some(s => s && s.length > 0)) {
                    playEntireBtn.disabled = false;
                }
            }
        }

        function handleAudioFrame(arrayBuffer) {
            if (!firstByteRecorded) {
                firstByteRecorded = true;
                statFirstByte.textContent = ((performance.now() - synthStart) / 1000).toFixed(2) + 's';
            }
            const int16 = new Int16Array(arrayBuffer);
            const float32 = new Float32Array(int16.length);
            for (let i = 0; i < int16.length; i++) float32[i] = int16[i] / 32768;
            currentChunkPieces.push(float32);
        }

        /**
         * Schedules a sentence:
         * 1. Safely trims boundary dead air without clipping speech tails.
         * 2. Plays each sentence fully to completion at 100% volume (no crossfade muting).
         * 3. Schedules the next sentence after a natural 100ms breathing pause so it
         *    never cuts off or talks over the previous sentence.
         * 4. Stores the trimmed sentence audio chunk for whole-paragraph replay.
         */
        function scheduleSentence(index) {
            if (currentChunkPieces.length === 0) {
                markChunk(index, 'done');
                return;
            }

            let totalLen = 0;
            for (const p of currentChunkPieces) totalLen += p.length;
            const merged = new Float32Array(totalLen);
            let offset = 0;
            for (const p of currentChunkPieces) { merged.set(p, offset); offset += p.length; }
            currentChunkPieces = [];

            // 1. Trim dead air padding from TTS output safely
            const trimmed = trimAudioSilence(merged, sampleRate);
            if (trimmed.length === 0) {
                markChunk(index, 'done');
                return;
            }

            // 1.5 Peak normalization: locks every sentence to a uniform peak level (0.80)
            // Eliminates volume jumps and prevents sentences from getting progressively louder!
            let maxPeak = 0;
            for (let i = 0; i < trimmed.length; i++) {
                const a = Math.abs(trimmed[i]);
                if (a > maxPeak) maxPeak = a;
            }
            if (maxPeak > 0.02) {
                const targetPeak = 0.80;
                const scale = targetPeak / maxPeak;
                const gain = Math.max(0.3, Math.min(2.5, scale));
                for (let i = 0; i < trimmed.length; i++) {
                    trimmed[i] *= gain;
                }
            }

            // Store trimmed chunk for "Play Entire Audio"
            storedSentences[index] = trimmed;

            const buffer = audioCtx.createBuffer(1, trimmed.length, sampleRate);
            buffer.getChannelData(0).set(trimmed);
            const duration = buffer.duration;

            const source = audioCtx.createBufferSource();
            source.buffer = buffer;

            const gainNode = audioCtx.createGain();
            source.connect(gainNode);
            gainNode.connect(analyser);

            // Natural pause between sentences (100ms): previous sentence finishes 100% cleanly,
            // then a natural breath pause, then next sentence begins.
            // NO crossfade muting or overlapping speech!
            const NATURAL_PAUSE_SEC = 0.10;
            const hasPrev = lastGainNode !== null;
            const targetStart = hasPrev
                ? (nextBoundary + NATURAL_PAUSE_SEC)
                : audioCtx.currentTime;

            const startAt = Math.max(targetStart, audioCtx.currentTime);

            gainNode.gain.setValueAtTime(1, startAt);
            source.start(startAt);

            // Synchronize reviewer UI highlight to actual acoustic speech time
            const delayToSpeak = Math.max(0, (startAt - audioCtx.currentTime) * 1000);
            const to = setTimeout(() => markChunk(index, 'speaking'), delayToSpeak);
            scheduledTimeouts.push(to);

            source.onended = () => {
                markChunk(index, 'done');
            };

            nextBoundary = startAt + duration;
            lastGainNode = gainNode;
            lastBufferDuration = duration;
            chunkLastNode[index] = source;
        }

        function renderChunkList(chunks) {
            chunkList.innerHTML = '';
            chunkLastNode = {};
            chunks.forEach((text, i) => {
                const el = document.createElement('div');
                el.className = 'chunk pending';
                el.id = 'chunk-' + i;
                el.innerHTML = `<div class="idx">${i + 1}</div><div class="text">${escapeHtml(text)}</div>`;
                chunkList.appendChild(el);
            });
        }

        function markChunk(index, state) {
            const el = document.getElementById('chunk-' + index);
            if (!el) return;
            el.className = 'chunk ' + state;
        }

        function escapeHtml(s) {
            return s.replace(/[&<>"']/g, (c) => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;' }[c]));
        }

        function stopEntireAudioPlayback() {
            if (entireAudioSource) {
                try { entireAudioSource.stop(); } catch (e) { }
                entireAudioSource = null;
            }
            entireAudioTimeouts.forEach(t => clearTimeout(t));
            entireAudioTimeouts = [];
        }

        function playEntireAudio() {
            const valid = storedSentences.filter(s => s && s.length > 0);
            if (valid.length === 0) return;

            ensureAudioContext();
            stopEntireAudioPlayback();

            // Inter-sentence natural pause (80ms)
            const pauseSamples = Math.floor(0.08 * sampleRate);
            let totalLength = 0;
            for (let i = 0; i < storedSentences.length; i++) {
                if (storedSentences[i]) {
                    totalLength += storedSentences[i].length;
                    if (i < storedSentences.length - 1) totalLength += pauseSamples;
                }
            }

            const merged = new Float32Array(totalLength);
            let offset = 0;
            const sentenceOffsets = [];

            for (let i = 0; i < storedSentences.length; i++) {
                if (storedSentences[i]) {
                    sentenceOffsets.push({ index: i, startSample: offset, length: storedSentences[i].length });
                    merged.set(storedSentences[i], offset);
                    offset += storedSentences[i].length;
                    if (i < storedSentences.length - 1) offset += pauseSamples;
                }
            }

            const buffer = audioCtx.createBuffer(1, merged.length, sampleRate);
            buffer.getChannelData(0).set(merged);

            const source = audioCtx.createBufferSource();
            source.buffer = buffer;
            source.connect(analyser); // sends audio through analyser to speakers

            entireAudioSource = source;
            playEntireBtn.textContent = '⏹ Stop entire audio';

            // Mark all chunks as pending before starting playback
            for (let i = 0; i < storedSentences.length; i++) {
                markChunk(i, 'pending');
            }

            const startTime = audioCtx.currentTime;
            source.start(startTime);

            // Sync visual reviewer highlights to each sentence as it plays
            sentenceOffsets.forEach(info => {
                const delayMs = (info.startSample / sampleRate) * 1000;
                const durMs = (info.length / sampleRate) * 1000;

                const startTo = setTimeout(() => markChunk(info.index, 'speaking'), delayMs);
                const endTo = setTimeout(() => markChunk(info.index, 'done'), delayMs + durMs);
                entireAudioTimeouts.push(startTo, endTo);
            });

            source.onended = () => {
                playEntireBtn.textContent = 'Play entire audio';
                entireAudioSource = null;
                for (let i = 0; i < storedSentences.length; i++) {
                    markChunk(i, 'done');
                }
            };
        }

        function synthesize() {
            if (!ws || ws.readyState !== WebSocket.OPEN) { alert('Not connected yet.'); return; }
            const rawText = inputText.value.trim();
            if (!rawText) return;

            ensureAudioContext();

            // Clear previous playback & stored entire-audio state
            stopEntireAudioPlayback();
            storedSentences = [];
            playEntireBtn.disabled = true;
            playEntireBtn.textContent = 'Play entire audio';

            scheduledTimeouts.forEach(t => clearTimeout(t));
            scheduledTimeouts = [];
            nextBoundary = audioCtx.currentTime;
            lastGainNode = null;
            lastBufferDuration = 0;
            currentChunkPieces = [];
            currentChunkIndex = -1;
            firstByteRecorded = false;
            synthStart = performance.now();
            statFirstByte.textContent = '—';
            statTotal.textContent = '—';

            // Auto-tag incoming paragraph: convert numbers & keywords to backend tags
            const processedText = autoTagText(rawText);
            updateTagPreview();

            // Send processed text to backend
            ws.send(JSON.stringify({ type: 'synthesize', text: processedText }));
        }

        connectBtn.addEventListener('click', connect);
        synthBtn.addEventListener('click', synthesize);
        playEntireBtn.addEventListener('click', () => {
            if (entireAudioSource) {
                stopEntireAudioPlayback();
                playEntireBtn.textContent = 'Play entire audio';
                for (let i = 0; i < storedSentences.length; i++) {
                    markChunk(i, 'done');
                }
            } else {
                playEntireAudio();
            }
        });
        setStatus('disconnected');
    </script>

</body>

</html>